In [ ]:
import pprint
import radiate as rd
import numpy as np
import polars as pl  # type: ignore

rd.random.seed(123)

In [8]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [9]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alters(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(rd.Limit.score(0.01), rd.Limit.generations(500))
)

result = engine.run(log=True)

2026-07-05T15:27:15.762344Z  INFO Epoch 1    | Score:   1.5998 | Time: 2.04ms
2026-07-05T15:27:15.763780Z  INFO Epoch 2    | Score:   1.1248 | Time: 3.20ms
2026-07-05T15:27:15.764823Z  INFO Epoch 3    | Score:   0.3618 | Time: 4.22ms
2026-07-05T15:27:15.765887Z  INFO Epoch 4    | Score:   0.3618 | Time: 5.13ms
2026-07-05T15:27:15.767162Z  INFO Epoch 5    | Score:   0.3618 | Time: 6.36ms
2026-07-05T15:27:15.768089Z  INFO Epoch 6    | Score:   0.3520 | Time: 7.27ms
2026-07-05T15:27:15.769030Z  INFO Epoch 7    | Score:   0.3520 | Time: 8.20ms
2026-07-05T15:27:15.769922Z  INFO Epoch 8    | Score:   0.3520 | Time: 9.08ms
2026-07-05T15:27:15.770812Z  INFO Epoch 9    | Score:   0.3520 | Time: 9.95ms
2026-07-05T15:27:15.771696Z  INFO Epoch 10   | Score:   0.3185 | Time: 10.83ms
2026-07-05T15:27:15.772586Z  INFO Epoch 11   | Score:   0.3185 | Time: 11.70ms
2026-07-05T15:27:15.773714Z  INFO Epoch 12   | Score:   0.3185 | Time: 12.81ms
2026-07-05T15:27:15.774618Z  INFO Epoch 13   | Score:   0.284

In [10]:
# collector.plot(
#     "count.species",
# )

In [11]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",80.0,40107.0,80.053894,0.913833,0.83509,0.0,80.0,100.0,501,null,null,null,null,null,null,499,1,"[""statistic""]"
"""step.evaluate.time""",0.00072,0.37663,0.000377,0.000378,1.4259e-7,0.0,8.3000e-8,0.001096,1000,376629µs,376µs,377µs,0µs,1096µs,0µs,499,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,10000.0,20.0,0.0,0.0,0.0,20.0,20.0,500,null,null,null,null,null,null,499,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000001,0.000727,0.000001,8.8219e-7,7.7826e-13,0.0,0.000001,0.000015,500,727µs,1µs,0µs,1µs,14µs,0µs,499,1,"[""selector"", ""time""]"
"""selector.boltzmann""",80.0,40000.0,80.0,0.0,0.0,0.0,80.0,80.0,500,null,null,null,null,null,null,499,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mutator.gaussian.rate""",0.3,150.0,0.3,0.0,0.0,0.0,0.3,0.3,500,null,null,null,null,null,null,499,1,"[""alterer"", ""mutator"", … ""expr""]"
"""step.metric.time""",0.000011,0.00606,0.000012,0.000003,6.4070e-12,0.0,0.00001,0.000046,500,6059µs,12µs,2µs,10µs,45µs,0µs,499,1,"[""time"", ""step""]"
"""time""",0.000843,0.441197,0.000882,0.000075,5.5867e-9,0.0,0.000811,0.002035,500,441196µs,882µs,74µs,810µs,2035µs,0µs,499,1,"[""time""]"


In [12]:
print(result.metrics().dashboard())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["pct.carryover"].to_dict())

[33 metrics, 16004 updates]
----- Metrics ----- (33 :: 16004) 
  carryover: 0.160  diversity: 0.953  unique_members: 95  unique_scores: 95  improvements: 500  iter_time(mean): 882.39µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.470      | 0.000      | 9.000      | 100    | 0.047        | 1.314      | 15.218     | 133.103   
genome.size                | dist   | 88.000     | 88.000     | 88.000     | 100    | 8.800        | 0.000      | 0.000      | 0.000     
scores                     | dist   | 1.802      | 0.075      | 6.804      | 100    | 0.180        | 1.646      | 2.251      | 9.400     


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | T